# Chronos-Bolt Forecast - Sales Demand
Zero-shot forecasting using `amazon/chronos-bolt-small` on retail inventory dataset.

In [8]:
!pip install -q git+https://github.com/amazon-science/chronos-forecasting.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [9]:
import pandas as pd
import numpy as np
import torch
import time
import matplotlib.pyplot as plt
from chronos import ChronosBoltPipeline

import os

MODEL_ID  = "amazon/chronos-bolt-small"
TARGET    = "Demand"
HORIZONS  = [7, 14, 28]
TRAIN_END = '2023-06-30'
VAL_END   = '2023-10-31'
LAG       = 7

# Auto-detect environment
if os.path.exists('/kaggle/input'):
    import glob
    matches = glob.glob('/kaggle/input/**/sales_data.csv', recursive=True)
    DATA_PATH = matches[0] if matches else '/kaggle/input/sales_data.csv'
    RESULT_DIR = '/kaggle/working'
else:
    DATA_PATH  = '/Users/P837032/Daily/Model/dataset/sales_data.csv'
    RESULT_DIR = '/Users/P837032/Daily/Model/result'

print(f'DATA_PATH: {DATA_PATH}')

DATA_PATH: /kaggle/input/datasets/atomicd/retail-store-inventory-and-demand-forecasting/sales_data.csv


## 1. Load Data

In [10]:
df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
df["series_id"] = df["Store ID"] + "_" + df["Product ID"]
print(f"Shape: {df.shape}")
print(f"Date range: {df['Date'].min()} - {df['Date'].max()}")
print(f"Series count: {df['series_id'].nunique()}")
df.head()

Shape: (76000, 17)
Date range: 2022-01-01 00:00:00 - 2024-01-30 00:00:00
Series count: 100


,Date,Store ID,Product ID,Category,Region,Inventory Level,Units Sold,Units Ordered,Price,Discount,Weather Condition,Promotion,Competitor Pricing,Seasonality,Epidemic,Demand,series_id
0,2022-01-01,S001,P0001,Electronics,North,195,102,252,72.72,5,Snowy,0,85.73,Winter,0,115,S001_P0001
1,2022-01-01,S001,P0002,Clothing,North,117,117,249,80.16,15,Snowy,1,92.02,Winter,0,229,S001_P0002
2,2022-01-01,S001,P0003,Clothing,North,247,114,612,62.94,10,Snowy,1,60.08,Winter,0,157,S001_P0003
3,2022-01-01,S001,P0004,Electronics,North,139,45,102,87.63,10,Snowy,0,85.19,Winter,0,52,S001_P0004
4,2022-01-01,S001,P0005,Groceries,North,152,65,271,54.41,0,Snowy,0,51.63,Winter,0,59,S001_P0005


## 2. Metrics

In [11]:
def mase(y, p, train):
    lag = min(LAG, len(train) - 1)
    denom = np.mean(np.abs(train[lag:] - train[:-lag]))
    return np.mean(np.abs(y - p)) / denom if denom > 0 else np.nan

## 3. Load Model

In [12]:
pipeline = ChronosBoltPipeline.from_pretrained(MODEL_ID, device_map="cpu", torch_dtype=torch.float32)
print('Model loaded.')

Model loaded.


## 4. Rolling Eval per Store × Product × Horizon

In [13]:
df['series_id'] = df['Store ID'] + '_' + df['Product ID']
series_ids = sorted(df['series_id'].unique())
os.makedirs(RESULT_DIR, exist_ok=True)

results = []
details = []

for h in HORIZONS:
    print(f'\n=== Horizon = {h} ===')
    scores = {'mase': []}

    # Build contexts từ VAL_END trở về trước (dùng làm train context)
    contexts, actuals_list, trains_list = [], [], []
    valid_ids = []
    eval_start = pd.Timestamp(VAL_END) + pd.Timedelta(days=1)

    for sid in series_ids:
        sdf = df[df['series_id'] == sid].sort_values('Date').set_index('Date')
        ts_train = sdf[:VAL_END][TARGET].values.astype(np.float32)
        ts_test  = sdf[eval_start:][TARGET].values.astype(np.float32)
        if len(ts_test) < h or len(ts_train) < 2:
            continue
        contexts.append(torch.tensor(ts_train))
        actuals_list.append(ts_test[:h])
        trains_list.append(ts_train)
        valid_ids.append(sid)

    start = time.time()
    forecasts = pipeline.predict(contexts, prediction_length=h)
    preds = forecasts.median(dim=1).values.numpy()
    print(f'  Inference: {time.time()-start:.1f}s for {len(valid_ids)} series')

    for i, sid in enumerate(valid_ids):
        store, product = sid.split('_', 1)
        y, p, tr = actuals_list[i], preds[i], trains_list[i]
        r = {'mase': mase(y, p, tr)}
        scores['mase'].append(r['mase'])
        details.append({'model': 'Chronos-Bolt-Small', 'store': store, 'product': product, 'horizon': h, 'mase': round(float(r['mase']), 4)})
        print(f"  {store} | {product} | MASE={r['mase']:.4f}")

    row = {
        'model': 'Chronos-Bolt-Small', 'dataset': 'retail_inventory_daily',
        'target': TARGET, 'horizon': h,
        'mean_mase':   round(float(np.nanmean(scores['mase'])), 4),
        'median_mase': round(float(np.nanmedian(scores['mase'])), 4),
    }
    results.append(row)
    print(f"  H={h} | MASE={row['mean_mase']:.4f} (med {row['median_mase']:.4f})")

summary = pd.DataFrame(results)
summary


=== Horizon = 7 ===
  Inference: 1.7s for 100 series
  S001 | P0001 | MASE=0.5955
  S001 | P0002 | MASE=0.5892
  S001 | P0003 | MASE=0.4135
  S001 | P0004 | MASE=0.4359
  S001 | P0005 | MASE=0.5889
  S001 | P0006 | MASE=0.5069
  S001 | P0007 | MASE=1.1844
  S001 | P0008 | MASE=0.6015
  S001 | P0009 | MASE=0.8137
  S001 | P0010 | MASE=0.6170
  S001 | P0011 | MASE=0.4551
  S001 | P0012 | MASE=0.4347
  S001 | P0013 | MASE=0.4930
  S001 | P0014 | MASE=0.5491
  S001 | P0015 | MASE=0.7953
  S001 | P0016 | MASE=0.5194
  S001 | P0017 | MASE=0.4047
  S001 | P0018 | MASE=0.6510
  S001 | P0019 | MASE=1.0188
  S001 | P0020 | MASE=0.4566
  S002 | P0001 | MASE=0.7496
  S002 | P0002 | MASE=0.6229
  S002 | P0003 | MASE=0.6858
  S002 | P0004 | MASE=0.7071
  S002 | P0005 | MASE=0.8428
  S002 | P0006 | MASE=0.8466
  S002 | P0007 | MASE=1.2721
  S002 | P0008 | MASE=0.5436
  S002 | P0009 | MASE=0.6623
  S002 | P0010 | MASE=0.8236
  S002 | P0011 | MASE=0.6877
  S002 | P0012 | MASE=0.7293
  S002 | P0013 | M

,model,dataset,target,horizon,mean_mase,median_mase
0,Chronos-Bolt-Small,retail_inventory_daily,Demand,7,0.7087,0.6739
1,Chronos-Bolt-Small,retail_inventory_daily,Demand,14,0.7159,0.6884
2,Chronos-Bolt-Small,retail_inventory_daily,Demand,28,0.7330,0.7209


## 5. Save & Compare

In [14]:
out_path = f'{RESULT_DIR}/chronos_bolt_daily_summary.csv'
summary.to_csv(out_path, index=False)
pd.DataFrame(details).to_csv(f'{RESULT_DIR}/chronos_bolt_daily_details.csv', index=False)
print(f'Saved to {out_path}')

# So sánh với baseline nếu có
baseline_path = f'{RESULT_DIR}/daily_baseline_summary.csv'
if os.path.exists(baseline_path):
    baseline = pd.read_csv(baseline_path)
    comparison = pd.concat([baseline, summary], ignore_index=True)
    display(comparison[['model','horizon','mean_mase','median_mase']])
else:
    display(summary[['model','horizon','mean_mase','median_mase']])

Saved to /kaggle/working/chronos_bolt_daily_summary.csv


,model,horizon,mean_mase,median_mase
0,Chronos-Bolt-Small,7,0.7087,0.6739
1,Chronos-Bolt-Small,14,0.7159,0.6884
2,Chronos-Bolt-Small,28,0.7330,0.7209
